In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

chroma_client = Chroma(
    persist_directory="/opt/chromadb/data",
    embedding_function=embedding_model
)

In [ ]:
def retrieve_context(query, chroma_client, top_k=5):
    results = chroma_client.similarity_search(query, k=top_k)
    context = "\n".join([r.page_content for r in results])
    return context

In [ ]:
def build_prompt(query, context):
    prompt = f"""You are Amber Support Assistant, an expert in Amber molecular dynamics software.

Context:
{context}

Question:
{query}

Answer the question clearly and concisely, using a step-by-step explanation when helpful."""
    return prompt

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import torch

class GoogleLLM:
    def __init__(
        self,
        model_name="google/flan-t5-large",
        max_new_tokens=300,
        temperature=0.7,
        use_device_map=False,
    ):
        self.model_name = model_name
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        # defaulting to CPU
        if use_device_map:
            self.pipe = pipeline(
                "text2text-generation",
                model=model_name,
                device_map="auto"
            )
        else:
            self.pipe = pipeline(
                "text2text-generation",
                model=model_name,
                device=0 if torch.cuda.is_available() else -1
            )

        self.gen_cfg = {
            "max_new_tokens": max_new_tokens,
            "do_sample": temperature > 0.0,
            "temperature": float(temperature) if temperature > 0.0 else None,
            "num_beams": 4 if temperature == 0.0 else 1,
        }

    def generate_answer(self, prompt: str) -> str:
        output = self.pipe(prompt, **self.gen_cfg)
        return output[0]["generated_text"]

KeyboardInterrupt: 

In [ ]:
def rag_pipeline(user_query):
    context = retrieve_context(user_query, chroma_client)
    prompt = build_prompt(user_query, context)
    answer = generate_answer(prompt)
    return answer

In [ ]:
if __name__ == "__main__":
    # temporary query to test if it will give a response
    query = "How does Amber handle time-series anomaly detection?"
    response = rag_pipeline(query)
    print("\nAMBER Support:\n", response)